# 3 · Searching a motif in hyperspace

In notebook 1, Cypher found the precedent for SO-1042 with an exact motif query: lamps, East
London depot, Ireland. In notebook 2, we met the three HDC operators. In this notebook we
answer the same question a second way, with hypervectors.

There's one design rule: **we don't encode orders, we encode edges.** Every relationship gets
its own hypervector, stored on its row in the relationship table, next to `src` and `dst`. A
motif is never stored. It's **composed on demand** from those edge hypervectors, for whatever
pattern the analyst asks about.

Why? Because the motif is the analyst's choice, and it changes. Today it's "lamps from East
London to Ireland". Tomorrow it might be "this exact product from this depot", or something
nobody has thought of yet. If we encoded one hypervector per order upfront, we'd bake today's
question into the data, and every new question would mean re-encoding every order. HDC's
operators are composable, so we don't have to.

What we'll see:

1. Each edge is encoded once, when it's written, independently of any question.
2. A path is composed from the edges along it: the graph walks, HDC composes.
3. HDC finds the same precedent, SO-0917, that Cypher found, and **ranks** every other order
   by how much of the motif it shares.
4. A Cypher query can produce the same ranking, if we spell the scoring out. One models the
   connections **implicitly**, the other **explicitly**.
5. A different motif, composed from the same stored edges, with nothing re-encoded.

## Three more terms

| Term | What it means |
|---|---|
| **edge hypervector** | One edge, encoded as its relationship type bound to the node it points to: $\mathbf{r} \otimes \rho(\mathbf{target})$. It's stored as a `vector` column on the edge's row. |
| **motif hypervector** | One order's pattern for one question: the paths the analyst picked, each composed from edge hypervectors, bundled together. It's built at query time and thrown away after. |
| **associative search** | Looking things up by *what they resemble*, not by an exact key. You give a partial or approximate pattern, and get back the candidates most similar to it. In HDC this is a nearest-neighbour search by cosine similarity, also called an *associative memory*. |

## Setup

`sales_orders.py` holds the graph from notebook 1 (`write_graph`, `edge_rows`, `graph_config`)
and the codebook from notebook 2 (`make_codebook`). The seed is fixed, so these are the same
hypervectors as in notebook 2. We'll open the graph engine once the edges have their
hypervectors.

In [1]:
from sales_orders import NODES, EDGES, write_graph, edge_rows, graph_config, make_codebook

import lancedb
import polars as pl
import torch
import torchhd
from lance_csr import CypherEngine

pl.Config.set_tbl_rows(20)  # print every row of our small tables
pl.Config.set_fmt_str_lengths(200)  # and the HAD_ISSUE notes in full

db = lancedb.connect("sales.lancedb")
write_graph(db)

codebook = make_codebook()
D = len(codebook["ireland"])

## Which paths describe an order?

Before encoding anything, we decide **what makes two orders comparable** for today's
question. That's a modelling choice, and it's best made with the people who handle the cases.
Our colleague knows three things about SO-1042, each a path out of the order:

| Path from the order | Hops | SO-1042's value |
|---|---|---|
| `-[:CONTAINS]->(:Product)-[:IN_CATEGORY]->(category)` | 2 | lamps |
| `-[:SHIPS_FROM]->(depot)` | 1 | east-london |
| `-[:SHIPS_TO]->(:Region)-[:IN_COUNTRY]->(country)` | 2 | ireland |

The region (Dublin) isn't in the question, because our colleague doesn't know it. The
customer and the exact product are left out on purpose too. This choice belongs to the
question, not to the data: nothing we store below depends on it.

## One path, one hypervector

What we want: a hypervector for a path, such as "ships to a region in Ireland", that is
different from every other path. Two paths that end at the same node must still differ if
they get there differently.

How: we treat the path as a key-value pair (notebook 2). The key is the chain of relationship
types, and the value is the node at the end. Permutation ρ marks each step's position along
the path, so the order of the hops counts:

$$\text{path}(r_1, \dots, r_k, n) = \mathbf{r_1} \otimes \rho(\mathbf{r_2}) \otimes \dots \otimes \rho^{k-1}(\mathbf{r_k}) \otimes \rho^k(\mathbf{n})$$

For example, $\text{path}(\text{SHIPS\_TO}, \text{IN\_COUNTRY}, \text{ireland}) = \mathbf{SHIPS\_TO} \otimes \rho(\mathbf{IN\_COUNTRY}) \otimes \rho^2(\mathbf{ireland})$.

In [2]:
def path(*steps):
    hv = codebook[steps[0]]
    for k, step in enumerate(steps[1:], start=1):
        hv = torchhd.bind(hv, torchhd.permute(codebook[step], shifts=k))
    return hv

to_ireland = path("SHIPS_TO", "IN_COUNTRY", "ireland")
to_ireland[:10].tolist()

[1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0]

### Roles stay separate

Here's why the relationship types belong in the key. Suppose the retailer opens a depot in
Dublin. Then an order could reach the `ireland` node in two **roles**: shipped *to* Ireland,
or shipped *from* Ireland. Both paths end at the same node, but their hypervectors are
unrelated, so a search for one won't match the other.

In [3]:
from_ireland = path("SHIPS_FROM", "IN_COUNTRY", "ireland")
print(f"cos(shipped to Ireland, shipped from Ireland) = {torchhd.cosine_similarity(to_ireland, from_ireland).item():+.3f}")

cos(shipped to Ireland, shipped from Ireland) = +0.003


## Store a hypervector on every edge

What we want: encode the graph's relationships once, in a form that *any* future motif can be
built from.

How: give every edge the hypervector of the one-hop path it makes, read from its source:

$$\mathbf{e} = \text{path}(r, \text{target}) = \mathbf{r} \otimes \rho(\mathbf{target})$$

We add it to each relationship table as a `vector` column. This is the only encoding step in
the notebook, and it works edge by edge: it doesn't know about orders, motifs or questions.
When a new edge is written later, it gets its own hypervector at the same time.

In [4]:
name = {n["id"]: n["name"] for rows in NODES.values() for n in rows}

for rel in EDGES:
    rows = [row | {"vector": path(rel, name[row["dst"]]).tolist()} for row in edge_rows(rel)]
    db.create_table(rel, rows, mode="overwrite")

db.open_table("SHIPS_FROM").search().limit(3).to_polars()

src,dst,vector
i64,i64,"array[f32, 10000]"
901,401,"[1.0, 1.0, … 1.0]"
904,403,"[-1.0, -1.0, … -1.0]"
908,402,"[1.0, 1.0, … 1.0]"


Now we open the graph engine over the tables. The `vector` column is just one more edge
property, like `note`, so the CSR adjacency is the same as in notebook 1.

In [5]:
engine = CypherEngine(graph_config(), {t: db.open_table(t).to_lance() for t in (*NODES, *EDGES)}, use_csr=True)


def cypher(q):
    return engine.execute(q)

## Compose a path from its edges

What we want: SO-1042's path "ships to a region in Ireland", built from the edges stored in the
graph, not from names.

How: permutation only moves entries around, so it distributes over binding:
$\rho(\mathbf{a} \otimes \mathbf{b}) = \rho(\mathbf{a}) \otimes \rho(\mathbf{b})$. That lets us
peel the path formula one hop at a time:

$$\text{path}(r_1, r_2, n) = \mathbf{r_1} \otimes \rho(\mathbf{r_2}) \otimes \rho^2(\mathbf{n}) = \mathbf{r_1} \otimes \rho\big(\mathbf{r_2} \otimes \rho(\mathbf{n})\big) = \mathbf{r_1} \otimes \rho(\mathbf{e_2})$$

where $\mathbf{e_2}$ is the hypervector stored on the path's **last edge**. So the graph walks
the path and returns that edge's vector, and HDC wraps the earlier hops' relationship types
around it. Longer paths repeat the same step.

In [6]:
q = "MATCH (o:Order {name: 'SO-1042'})-[:SHIPS_TO]->()-[e:IN_COUNTRY]->() RETURN e.vector AS vector"
e2 = torch.tensor(cypher(q)["vector"][0].as_py())
composed = torchhd.bind(codebook["SHIPS_TO"], torchhd.permute(e2, shifts=1))

similarity = torchhd.cosine_similarity(composed, path("SHIPS_TO", "IN_COUNTRY", "ireland")).item()
print(f"cos(composed from edges, path from names) = {similarity:.3f}")
assert similarity > 0.999

cos(composed from edges, path from names) = 1.000


The composed vector is exactly the path hypervector. Notice that it doesn't mention Dublin:
the middle node of a path drops out, just as in the path formula. The path says *where it
leads*, not *which way it went*.

## Compose a motif for every past order

What we want: for a given question, one motif hypervector per past order with an issue. These
are the candidates we'll search.

How: two small functions.

- `path_vectors(*rels)` walks one path for every past order with a single Cypher query,
  returns the stored vector of each path's last edge, and composes it as above.
- `compose_motifs(*paths)` does that for each path in the question, and bundles each order's
  paths into its motif hypervector.

Nothing is written to the database. The motifs live in memory for this question only.

In [7]:
def path_vectors(*rels):
    """Each past order's path along rels, composed from the vector stored on the path's last edge."""
    hops = "".join(f"-[:{r}]->()" for r in rels[:-1])
    q = f"MATCH (o:Order){hops}-[e:{rels[-1]}]->(), (o)-[:HAD_ISSUE]->() RETURN o.name AS order_id, e.vector AS vector"
    vectors = {}
    for row in cypher(q).to_pylist():
        hv = torch.tensor(row["vector"])
        for r in reversed(rels[:-1]):
            hv = torchhd.bind(codebook[r], torchhd.permute(hv, shifts=1))
        vectors[row["order_id"]] = hv
    return vectors


def compose_motifs(*paths):
    """One motif hypervector per past order: the bundle of its composed paths."""
    per_path = [path_vectors(*rels) for rels in paths]
    orders = sorted(set().union(*per_path))
    return {o: torchhd.multiset(torch.stack([vectors[o] for vectors in per_path if o in vectors])) for o in orders}

## Search with what we know

A question is a list of paths, each ending at the node we're asking about. The same list gives
both sides of the search:

- the **query**: each path built from the codebook by name, then bundled;
- the **candidates**: the same paths, without their end nodes, composed from every past
  order's edges.

Then we score every candidate by cosine similarity and rank them.

![A query hypervector compared with three candidate hypervectors, with one match highlighted](../img/hdc-retrieval.png)

*The retrieval branch from slide 13. The same selected paths describe the query and candidates; cosine similarity ranks the past cases. Candidate motifs are composed from stored edge vectors for this question. The coloured patterns are illustrative.*

In [8]:
def search(question):
    query = torchhd.multiset(torch.stack([path(*steps) for steps in question]))
    candidates = compose_motifs(*(steps[:-1] for steps in question))
    scores = [
        {"order_id": order, "score": round(torchhd.cosine_similarity(query, motif).item(), 3)}
        for order, motif in candidates.items()
    ]
    return pl.DataFrame(scores).sort(["score", "order_id"], descending=[True, False])


question = [
    ("CONTAINS", "IN_CATEGORY", "lamps"),
    ("SHIPS_FROM", "east-london"),
    ("SHIPS_TO", "IN_COUNTRY", "ireland"),
]
hits = search(question)
hits

order_id,score
str,f64
"""SO-0917""",1.0
"""SO-0901""",0.669
"""SO-0920""",0.667
"""SO-0903""",0.342
"""SO-0907""",0.342
"""SO-0904""",0.338
"""SO-0908""",0.338
"""SO-0912""",0.335
"""SO-0906""",0.025


In [9]:
assert hits["order_id"][0] == "SO-0917" and hits["score"][0] > 0.95
assert (hits["score"][1:] < 0.7).all()

SO-0917 comes first, the same precedent Cypher found. Notice the scores come in **tiers**:
about 1, then 0.67, then 0.33, then 0. Every order in a tier shares the same number of the
question's three paths:

- **1**: all three (SO-0917);
- **0.67**: two of them (SO-0901 and SO-0920 also ship from East London to Ireland, but aren't lamps);
- **0.33**: one;
- **0**: none.

### Why those numbers?

A path that the query and an order share contributes $D$ to their dot product. A path that
only one of them has contributes noise near $0$. The query and every candidate bundle the same
3 paths, so both have length about $\sqrt{3D}$. For $m$ shared paths:

$$\cos(\mathbf{q}, \mathbf{M}) \approx \frac{m \cdot D}{\sqrt{3D}\,\sqrt{3D}} = \frac{m}{3} \qquad m = 3, 2, 1, 0 \;\Rightarrow\; 1,\ 0.67,\ 0.33,\ 0$$

The region never entered the computation: we composed only the paths in the question. With a
motif encoded upfront, we'd have had to decide in advance whether the region was part of it.

## The same search, written explicitly

Now we run the same search in Cypher. First, the exact motif query from notebook 1. It
returns exactly HDC's top tier.

In [10]:
exact = '''
MATCH (o:Order)-[:CONTAINS]->(:Product)-[:IN_CATEGORY]->(:Category {name: 'lamps'}),
      (o)-[:SHIPS_FROM]->(:Depot {name: 'east-london'}),
      (o)-[:SHIPS_TO]->(:Region)-[:IN_COUNTRY]->(:Country {name: 'ireland'}),
      (o)-[:HAD_ISSUE]->(:Issue)
RETURN o.name AS order_id
'''
exact_ids = set(cypher(exact).column("order_id").to_pylist())
top_tier = set(hits.filter(pl.col("score") > 0.9)["order_id"])
print("Cypher:", exact_ids, "  HDC top tier:", top_tier)
assert exact_ids == top_tier

Cypher: {'SO-0917'}   HDC top tier: {'SO-0917'}


To get the near misses as well, Cypher needs us to **spell out the scoring**: match every
order's paths, and count how many of the three conditions hold, with one `CASE` per
condition.

In [11]:
graded = '''
MATCH (o:Order)-[:CONTAINS]->(:Product)-[:IN_CATEGORY]->(c:Category),
      (o)-[:SHIPS_FROM]->(d:Depot),
      (o)-[:SHIPS_TO]->(:Region)-[:IN_COUNTRY]->(k:Country),
      (o)-[:HAD_ISSUE]->(:Issue)
RETURN o.name AS order_id,
       (CASE WHEN c.name = 'lamps' THEN 1 ELSE 0 END)
     + (CASE WHEN d.name = 'east-london' THEN 1 ELSE 0 END)
     + (CASE WHEN k.name = 'ireland' THEN 1 ELSE 0 END) AS matches
ORDER BY matches DESC, order_id
'''
counts = pl.from_arrow(cypher(graded))
counts

order_id,matches
str,i64
"""SO-0917""",3
"""SO-0901""",2
"""SO-0920""",2
"""SO-0903""",1
"""SO-0904""",1
"""SO-0907""",1
"""SO-0908""",1
"""SO-0912""",1
"""SO-0906""",0


Put the two side by side. For every order, the HDC score is the Cypher count divided by 3,
give or take the $\pm 0.01$-sized noise from notebook 2. Same ranking, two methods.

In [12]:
side_by_side = counts.join(hits, on="order_id", maintain_order="left").with_columns(
    (pl.col("matches") / 3).round(3).alias("matches / 3")
)
side_by_side

order_id,matches,score,matches / 3
str,i64,f64,f64
"""SO-0917""",3,1.0,1.0
"""SO-0901""",2,0.669,0.667
"""SO-0920""",2,0.667,0.667
"""SO-0903""",1,0.342,0.333
"""SO-0904""",1,0.338,0.333
"""SO-0907""",1,0.342,0.333
"""SO-0908""",1,0.338,0.333
"""SO-0912""",1,0.335,0.333
"""SO-0906""",0,0.025,0.0


In [13]:
assert ((side_by_side["score"] - side_by_side["matches / 3"]).abs() < 0.05).all()

## A new question, from the same edges

Now our colleague asks something narrower: "has **this exact product**, the cordless table
lamp, gone wrong when shipped from East London?" That's a different motif. It uses the
product itself, one hop away, instead of its category, and it ignores the destination
entirely.

If we'd stored one motif per order, this question would need every order re-encoded. Here
it's just a new list of paths. The `CONTAINS` edges' own hypervectors weren't used by the
first question at all (there, `CONTAINS` was only the first hop of the category path), but
they've been in the table since we wrote the edges.

In [14]:
new_question = [
    ("CONTAINS", "cordless-table-lamp"),
    ("SHIPS_FROM", "east-london"),
]
by_product = search(new_question)
by_product

order_id,score
str,f64
"""SO-0917""",1.0
"""SO-0901""",0.495
"""SO-0920""",0.495
"""SO-0903""",0.016
"""SO-0908""",0.014
"""SO-0914""",0.014
"""SO-0915""",0.007
"""SO-0906""",0.003
"""SO-0904""",0.001


In [15]:
assert by_product["order_id"][0] == "SO-0917" and by_product["score"][0] > 0.95

SO-0917 is the only past order that shares both paths. The other East London orders share
one of two, so they score about $1/2$. Same stored edges, different question, no re-encoding.

## Implicit and explicit

Both methods answered the same questions about connected data. The difference is where the
connections live.

| | Graph + Cypher | Hypervectors |
|---|---|---|
| Connections are | **explicit**: each edge is a row you can point to | **implicit**: ⊗ and ρ fold a path's hops into one hypervector, and ⊕ folds the paths into a motif |
| A pattern is | a query you write | a hypervector you compose, the same way for the query and the candidates |
| Partial matches | need a scoring query, written for this question | come for free, as the similarity score |
| Answers are | exact and auditable | approximate (noise of about $1/\sqrt{D}$), ranked |
| Best at | "show me exactly what happened" | "show me what this resembles" |

In this notebook they also worked together. The graph walked each order's paths, through the
CSR, and HDC composed what it walked into one hypervector to compare. Neither side
precomputed the question.

That flexibility has a price. Because the candidates are composed at query time, we score
them directly instead of through a vector index. On a large graph, you'd let the graph narrow
the candidates first (say, past orders from the last year) and compose only those.

If our colleague had wanted to weigh the depot above the category, the Cypher scoring query
would need new weights. In HDC, you put the depot path into the question twice.

## From a candidate to the evidence

Similarity tells our colleague *which* past cases to look at, not what happened in them. For
that, we go back to the graph and read the `HAD_ISSUE` notes of the two top tiers.

In [16]:
shortlist = hits.filter(pl.col("score") > 0.5)["order_id"].to_list()
# An f-string is fine for names we produced ourselves; never build Cypher from user input this way.
q = f'''
MATCH (o:Order)-[h:HAD_ISSUE]->(i:Issue)
WHERE o.name IN {shortlist}
RETURN o.name, i.name, h.note
'''
pl.from_arrow(cypher(q))

o.name,i.name,h.note
str,str,str
"""SO-0901""","""customs_delay""","""Commercial invoice missing; Irish customs held the parcel until the broker resent it."""
"""SO-0917""","""dispatch_hold""","""Cordless lamps (lithium batteries) held at East London depot for dangerous-goods paperwork; released after calling the depot."""
"""SO-0920""","""customs_delay""","""Customs declaration lodged late by the broker."""


The exact match points to a **dispatch hold** at the East London depot. The two near misses
are **customs delays**, a different but plausible lead for an order going to Ireland. Our
colleague now knows to call the depot first, and to check the customs paperwork if that comes
up empty.

## Wrap-up

We encoded **edges, not orders**: each relationship got one hypervector,
$\mathbf{r} \otimes \rho(\mathbf{target})$, stored on its row in the relationship table. A
question became a list of paths. For each one, the graph walked every past order's path and
returned its last edge's hypervector, and ⊗ and ρ wrapped the earlier hops around it. ⊕
bundled the paths into a motif hypervector, composed on demand and never stored.

An associative search with a partial pattern found the same precedent as the exact Cypher
motif query, and ranked every other order by how much of the pattern it shares. A Cypher
scoring query reproduced that ranking, so the two methods agree; one stores the connections
implicitly and the other explicitly. And a different question, about the exact product, was
answered from the same stored edges without re-encoding anything.

It turns out SO-1042 was held at the depot too. In notebook 4 we go one step further: instead
of looking up individual cases, we summarise all past cases by issue type, and see what the
memory learns from SO-1042 alone.